# 🎧 Audio Tabular Agent Training (XGBoost, LightGBM, CatBoost)

This unified notebook trains **three high-performance gradient boosting models** (XGBoost, LightGBM, and CatBoost) on both **Spectral** and **Prosodic** speech features extracted from the ASVspoof 5 dataset.

### Features:
1. **Unified Pipeline**: Trains and evaluates models for both Spectral and Prosodic features.
2. **Automatic Dev Splitting**: Handles missing `dev` CSV files automatically.
3. **Triple Booster Strategy**: Trains and benchmarks **XGBoost**, **LightGBM**, and **CatBoost** using CUDA GPU-acceleration.
4. **Detailed Class Imbalance Analysis**: Live distribution tracking and safe **SMOTE** oversampling.
5. **Explainability & Diagnostics**: Compares the top 15 feature importances across all three models to see what acoustic features are most predictive of synthetic/deepfake speech.

In [ ]:
# Install all required booster and visualization libraries
!pip install -q xgboost lightgbm catboost pandas numpy scikit-learn matplotlib scipy joblib imbalanced-learn seaborn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import json
import subprocess
import xgboost as xgb
import lightgbm as lgb
import catboost as cb
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.metrics import roc_curve, accuracy_score, f1_score, auc as sklearn_auc, classification_report, confusion_matrix, precision_recall_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d

BASE_DIR = Path('/content/drive/MyDrive/142_Feature_Extracted')

# Auto-detect GPU
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    has_gpu = result.returncode == 0
except:
    has_gpu = False

print(f"GPU Detected: {has_gpu}")

In [ ]:
# Helper function to calculate Equal Error Rate (EER)
def calculate_eer(y_true, probas):
    fpr, tpr, thresholds = roc_curve(y_true, probas, pos_label=1)
    fnr = 1 - tpr
    try:
        eer = brentq(lambda x: interp1d(fpr, fnr - fpr)(x), 0, 1)
    except:
        # Fallback if brentq fails
        eer = float(np.mean(np.abs(fnr - fpr)))
    return eer, fpr, tpr

In [ ]:
def train_and_evaluate_agent(agent_name):
    DATA_DIR = BASE_DIR / agent_name / 'Dataset'
    MODEL_DIR = BASE_DIR / agent_name / 'Model'
    OUT_DIR = MODEL_DIR / 'saved_model'
    RESULTS_DIR = MODEL_DIR / 'Results'
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    
    print(f"\n{'='*70}\n🚀 STARTING TRAINING PIPELINE FOR: {agent_name.upper()} AGENT\n{'='*70}")
    
    # 1. Load Data
    train_csv = DATA_DIR / f'{agent_name}_train.csv'
    eval_csv = DATA_DIR / f'{agent_name}_eval.csv'
    dev_csv = DATA_DIR / f'{agent_name}_dev.csv'
    
    if not train_csv.exists() or not eval_csv.exists():
        print(f"❌ ERROR: Missing training or evaluation CSV for {agent_name}!")
        return
        
    train_df = pd.read_csv(train_csv)
    test_df = pd.read_csv(eval_csv)
    
    if dev_csv.exists():
        dev_df = pd.read_csv(dev_csv)
    else: 
        print("⚠️ dev.csv not found! Automatically carving 10% Dev set from Train...")
        if train_df['label'].nunique() > 1:
            train_df, dev_df = train_test_split(train_df, test_size=0.1, random_state=42, stratify=train_df['label'])
        else:
            train_df, dev_df = train_test_split(train_df, test_size=0.1, random_state=42)
            
    print(f"📊 Data Splits -> Train: {len(train_df):,} | Dev: {len(dev_df):,} | Eval: {len(test_df):,}")
    
    # 2. Class Imbalance Analysis
    counts = train_df['label'].value_counts().sort_index()
    total = len(train_df)
    print(f"\n⚖️ Class Distribution:")
    for label, count in counts.items():
        class_name = "Bonafide" if label == 0 else "Spoof"
        print(f"  - Class {label} [{class_name}]: {count:,} ({count/total*100:.2f}%)")
        
    # 3. Features Prep & Safe SMOTE
    META_COLS = {'label', 'filename', 'split'}
    feat_cols = [c for c in train_df.columns if c not in META_COLS]
    
    X_train = train_df[feat_cols].values.astype(np.float32)
    y_train = train_df['label'].values
    X_dev   = dev_df[feat_cols].values.astype(np.float32)
    y_dev   = dev_df['label'].values
    X_test  = test_df[feat_cols].values.astype(np.float32)
    y_test  = test_df['label'].values
    
    np.nan_to_num(X_train, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    np.nan_to_num(X_dev,   copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    np.nan_to_num(X_test,  copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    
    n_unique_labels = len(np.unique(y_train))
    if n_unique_labels > 1:
        print("Applying SMOTE to perfectly balance classes (1:1)... ")
        smote = SMOTE(random_state=42)
        X_train, y_train = smote.fit_resample(X_train, y_train)
        print(f"  - Balanced Train Shape: {X_train.shape} (Spoof: {(y_train==1).sum():,}, Bonafide: {(y_train==0).sum():,})")
    else:
        print("⚠️ Skipping SMOTE: Only 1 class found in labels.")
        
    # 4. Define and Train Boosters
    xgb_method = 'gpu_hist' if has_gpu else 'hist'
    xgb_device = 'cuda' if has_gpu else 'cpu'
    lgb_device = 'gpu' if has_gpu else 'cpu'
    cb_device = 'GPU' if has_gpu else 'CPU'
    
    # --- A. XGBOOST ---
    print("\n--- 🪵 Training XGBoost ---")
    model_xgb = xgb.XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
        eval_metric=['logloss', 'aucpr'], early_stopping_rounds=50, tree_method=xgb_method, device=xgb_device, n_jobs=-1
    )
    model_xgb.fit(X_train, y_train, eval_set=[(X_dev, y_dev)], verbose=100)
    
    # --- B. LIGHTGBM ---
    print("\n--- ⚡ Training LightGBM ---")
    model_lgb = lgb.LGBMClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
        device=lgb_device, n_jobs=-1, verbose=-1
    )
    model_lgb.fit(X_train, y_train, eval_set=[(X_dev, y_dev)], callbacks=[lgb.early_stopping(50, verbose=False)])
    
    # --- C. CATBOOST ---
    print("\n--- 🐈 Training CatBoost ---")
    model_cb = cb.CatBoostClassifier(
        iterations=500, depth=6, learning_rate=0.05, eval_metric='AUC', early_stopping_rounds=50,
        task_type=cb_device, verbose=100
    )
    model_cb.fit(X_train, y_train, eval_set=(X_dev, y_dev))
    
    # 5. Evaluate and Compare Models
    models = {"XGBoost": model_xgb, "LightGBM": model_lgb, "CatBoost": model_cb}
    metrics = {}
    
    plt.figure(figsize=(10, 8))
    
    for name, model in models.items():
        probas = model.predict_proba(X_test)[:, 1]
        preds = (probas >= 0.5).astype(int)
        
        eer, fpr, tpr = calculate_eer(y_test, probas)
        auc = sklearn_auc(fpr, tpr)
        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds)
        
        metrics[name] = {"EER": eer, "AUC": auc, "Accuracy": acc, "F1": f1}
        
        print(f"\n📌 Model Performance: {name}")
        print(f"  - EER      : {eer*100:.2f}%")
        print(f"  - AUC      : {auc:.4f}")
        print(f"  - Accuracy : {acc*100:.2f}%")
        
        plt.plot(fpr, tpr, label=f"{name} (EER: {eer*100:.2f}% | AUC: {auc:.3f})")
        
        # Save Model to Google Drive
        if name == "XGBoost":
            model.save_model(str(OUT_DIR / f'best_{agent_name}_xgb.json'))
        elif name == "LightGBM":
            model.booster_.save_model(str(OUT_DIR / f'best_{agent_name}_lgb.txt'))
        elif name == "CatBoost":
            model.save_model(str(OUT_DIR / f'best_{agent_name}_cat.json'))
            
    plt.plot([0, 1], [0, 1], '--', color='gray')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ASVspoof 5 - {agent_name.capitalize()} ROC Curve Comparison')
    plt.legend()
    plt.savefig(str(RESULTS_DIR / f'{agent_name}_roc_comparison.png'), dpi=150)
    plt.show()
    
    # Save metrics report
    with open(RESULTS_DIR / 'metrics.json', 'w') as f:
        json.dump(metrics, f, indent=4)
    with open(OUT_DIR / 'feature_cols.json', 'w') as f:
        json.dump(feat_cols, f)
        
    # 6. EXPLAINABILITY: Plot Feature Importances Comparison
    print(f"\n{'='*70}\n🧠 ACOUSTIC EXPLAINABILITY & DIAGNOSTICS\n{'='*70}")
    fig, axes = plt.subplots(1, 3, figsize=(24, 7), sharey=False)
    
    # A. XGBoost Feature Importance
    xgb_imp = pd.DataFrame({'feature': feat_cols, 'importance': model_xgb.feature_importances_})
    xgb_imp = xgb_imp.sort_values('importance', ascending=False).head(15)
    sns.barplot(x='importance', y='feature', data=xgb_imp, ax=axes[0], palette='magma')
    axes[0].set_title("XGBoost Feature Importance (Top 15)")
    
    # B. LightGBM Feature Importance
    lgb_imp = pd.DataFrame({'feature': feat_cols, 'importance': model_lgb.feature_importances_})
    lgb_imp = lgb_imp.sort_values('importance', ascending=False).head(15)
    sns.barplot(x='importance', y='feature', data=lgb_imp, ax=axes[1], palette='viridis')
    axes[1].set_title("LightGBM Feature Importance (Top 15)")
    
    # C. CatBoost Feature Importance
    cb_imp = pd.DataFrame({'feature': feat_cols, 'importance': model_cb.get_feature_importance()})
    cb_imp = cb_imp.sort_values('importance', ascending=False).head(15)
    sns.barplot(x='importance', y='feature', data=cb_imp, ax=axes[2], palette='mako')
    axes[2].set_title("CatBoost Feature Importance (Top 15)")
    
    plt.tight_layout()
    plt.savefig(str(RESULTS_DIR / f'{agent_name}_explainability.png'), dpi=150)
    plt.show()
    
    print(f"🎉 Complete evaluation metrics, ROC curves, and Explainability analysis saved to Drive!")

In [ ]:
# Execute Spectral Agent Pipeline
train_and_evaluate_agent('spectral')

In [ ]:
# Execute Prosodic Agent Pipeline
train_and_evaluate_agent('prosodic')